In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../'))
sys.path.append(base_path)

import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard

import numpy as np
from tqdm import tqdm
import datetime

from data_loaders.data_loader_complete import DataLoaderComplete
import project_images_loader as pil

import discriminator as Discriminator
import generator as Generator
import feature_extractor as feature_extractor

import utils.functions as fc
import utils.util_deep_learning as udl
import utils.util as util
import utils.visualization as vis

tf.keras.backend.set_image_data_format("channels_first")
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')






TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.10.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported configuration, either change the TensorFlow version or the TensorFlow Addons's version. 
You can find the compatibility matrix in TensorFlow Addon's readme:
https://github.com/tensorflow/

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3060 Laptop GPU, compute capability 8.6


## Visualize data

In [2]:
# %matplotlib widget

data_train = DataLoaderComplete(image_sufix=pil.INAME_PREPROCESSED,
                                batch_size=1, 
                                image_size=(192, 224, 192), 
                                channels_first=True,
                                shuffle=True, 
                                nb_patients=2, 
                                shuffle_before_selecting=False, 
                                server=False,
                                start_from=10)

Fe = feature_extractor.create_features_extractor((1, 192, 224, 192), pil.get_genesis_weights_path_name())
# fc.imgshow_3D("", data_train[0][0][0,0])

feats = []


for i, (img_hunt3, img_hunt4) in enumerate(data_train):
    feats_img = Fe(img_hunt3, training=False)
    _feat = []
    for feat in feats_img:
        _feat.append(feat.numpy())
    feats.append(_feat)
    print(f"done for img {i}")
    del feats_img


done for img 0
done for img 1


In [3]:
print(feats[0][0].shape)
print(feats[0][1].shape)
print(feats[0][2].shape)
print(feats[0][3].shape)


(1, 64, 192, 224, 192)
(1, 64, 96, 112, 96)
(1, 256, 48, 56, 48)
(1, 512, 24, 28, 24)


In [4]:
feat_1 = np.zeros((len(feats), *feats[0][0].shape))
feat_2 = np.zeros((len(feats), *feats[0][1].shape))
feat_3 = np.zeros((len(feats), *feats[0][2].shape))
feat_4 = np.zeros((len(feats), *feats[0][3].shape))

for i, img_feats in enumerate(feats):
    feat_1[i] = img_feats[0]
    feat_2[i] = img_feats[1]
    feat_3[i] = img_feats[2]
    feat_4[i] = img_feats[3]


feat_1 = feat_1.reshape(len(feats), -1)
feat_2 = feat_2.reshape(len(feats), -1)
feat_3 = feat_3.reshape(len(feats), -1)
feat_4 = feat_4.reshape(len(feats), -1)



In [5]:
print(np.sum(feat_1[0] - feat_1[1]))
print(np.sum(feat_2[0] - feat_2[1]))
print(np.sum(feat_3[0] - feat_3[1]))
print(np.sum(feat_4[0] - feat_4[1]))

# 360205.3009697199
# 63300.56828254461
# 31939.455354213715
# 154962.47660946846

# 375277.9370186925
# 81266.0332890153
# 80309.95594418049
# 126822.14976972342

-375277.9370186925
-81266.0332890153
-80309.95594418049
-126822.14976972342
